# Full-history exploratory data analysis: Korneuburg (207241-at)

This notebook describes the data foundation for hourly water-level forecasting at
the **Donau / Korneuburg** gauge. It combines PegelAlarm gauge observations with
GeoSphere Austria INCA precipitation and 2 m air temperature, then audits the
preprocessing and feature-engineering artifacts used by the modeling pipeline.

The analysis asks four questions:

1. What period is covered, and where are the missing or interpolated observations?
2. How do water level, rainfall, and temperature vary over time and season?
3. What characterizes threshold exceedances and the rainfall preceding them?
4. Are the stored predictors and 24-hour target vectors statistically healthy and
   suitable for model development?

All stage-2 and stage-3 rows are concatenated in timestamp order and treated as one
history. No partition label is added, and no partition comparison is made. The
feature files themselves are kept exactly as stored; their single internal
generation seam is recorded only because independent calculation on either side
creates expected lag, rolling, and target nulls near that boundary.

**Units.** Water level and the catalog threshold are centimetres (cm),
precipitation is millimetres per hour (mm), and temperature is degrees Celsius
(°C). All timestamps and calendar summaries use UTC.

> **Interpretation caveat:** combining the complete history is appropriate for
> descriptive EDA, but results from this notebook are not blind test evidence and
> must not be reported as out-of-sample model performance.

## 1. Configuration and reproducible setup

Configuration is deliberately local to this notebook, except for the station
identifier and feature contract, which are imported from `src` so this notebook
cannot drift from the real pipeline's configuration. The notebook is read-only
with respect to every source artifact.

In [ ]:
from __future__ import annotations

import hashlib
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf, adfuller, kpss, pacf

from src.config import TARGET_STATION_ID

warnings.filterwarnings(
    "ignore",
    message="The 'generic' unit for NumPy timedelta is deprecated",
    category=DeprecationWarning,
)

STATION_ID = TARGET_STATION_ID
RAW_DIR = Path("data/raw")
JOINED_DIR = Path("data/processed/joined")
MAX_INTERPOLATION_GAP_HOURS = 6
RELATIONSHIP_WINDOWS = (6, 24, 72, 168)
RELATIONSHIP_HORIZONS = (1, 6, 12, 24)
EVENT_CONTEXT_DAYS = 7
TOP_EVENT_COUNT = 5

PATHS = {
    "station_catalog": RAW_DIR / "pegelalarm_stations_at.parquet",
    "raw_water": RAW_DIR / f"pegelalarm_{STATION_ID}_height_hour.parquet",
    "raw_weather": RAW_DIR / f"geosphere_inca_{STATION_ID}_hour.parquet",
    "preprocess_manifest": JOINED_DIR / "all_stations_preprocess_metadata.json",
    "feature_manifest": JOINED_DIR / "all_stations_feature_metadata.json",
}

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

PLOT_TEMPLATE = "plotly_white"
PLOT_COLORS = {
    "water": "#176B87",
    "rain": "#3FA7D6",
    "temperature": "#D95F59",
    "imputed": "#F2A900",
    "missing": "#6C757D",
    "threshold": "#B22222",
}

## 2. Load and validate the complete artifact chain

The checks below fail fast on missing files, hash drift, schema drift, timestamp
errors, row misalignment, or a violated predictor/target contract. They also prove
that raw observations and weather values are preserved in the processed timeline.
Successful execution is therefore an auditable prerequisite for every later chart.

In [ ]:
from src.feature_engineering import (
    DEFAULT_FEATURE_CONFIG,
    extract_station_frame,
    feature_column_names,
    target_column_names,
)


def sha256(path: Path) -> str:
    """Return the SHA-256 digest for a file without modifying it."""
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def utc_text(value: pd.Timestamp) -> str:
    """Format a timezone-aware timestamp in manifest-compatible UTC text."""
    return pd.Timestamp(value).isoformat().replace("+00:00", "Z")


def assert_utc_ordered_unique(
    frame: pd.DataFrame,
    timestamp_column: str,
    *,
    hourly: bool,
) -> None:
    """Assert UTC dtype, chronological order, uniqueness, and optional continuity."""
    series = frame[timestamp_column]
    dtype = series.dtype
    assert isinstance(dtype, pd.DatetimeTZDtype) and str(dtype.tz) == "UTC"
    assert series.is_monotonic_increasing
    assert not series.duplicated().any()
    if hourly:
        expected = pd.date_range(series.iloc[0], periods=len(series), freq="h")
        assert pd.DatetimeIndex(series).equals(expected)


def validate_profile(profile: dict, frame: pd.DataFrame) -> None:
    """Validate a loaded frame against one manifest profile."""
    path = Path(profile["path"])
    assert path.is_file()
    assert sha256(path) == profile["sha256"]
    assert len(frame) == profile["rows"]
    assert {column: str(dtype) for column, dtype in frame.dtypes.items()} == (
        profile["schema"]
    )
    assert (
        utc_text(frame["timestamp"].iloc[0])
        == (profile["timestamp_range"]["start_utc"])
    )
    assert (
        utc_text(frame["timestamp"].iloc[-1]) == (profile["timestamp_range"]["end_utc"])
    )
    if "null_counts" in profile:
        actual_nulls = {
            column: int(count) for column, count in frame.isna().sum().items()
        }
        assert actual_nulls == profile["null_counts"]


def target_eligibility(frame: pd.DataFrame, horizon: int) -> pd.Series:
    """Reproduce the recorded all-observed future-target eligibility rule."""
    observed = frame["water_level"].notna() & ~frame["imputed"]
    future_observed = observed.shift(-1)
    valid_count = (
        future_observed.iloc[::-1]
        .rolling(horizon, min_periods=horizon)
        .sum()
        .iloc[::-1]
    )
    return valid_count.eq(horizon)


for path in PATHS.values():
    assert path.is_file(), f"Required input is missing: {path}"

preprocess_manifest = json.loads(
    PATHS["preprocess_manifest"].read_text(encoding="utf-8")
)
feature_manifest = json.loads(PATHS["feature_manifest"].read_text(encoding="utf-8"))
station_catalog = pd.read_parquet(PATHS["station_catalog"])
raw_water = (
    pd.read_parquet(PATHS["raw_water"]).sort_values("sourceDate").reset_index(drop=True)
)
raw_weather = (
    pd.read_parquet(PATHS["raw_weather"]).sort_values("time").reset_index(drop=True)
)

source_profiles = sorted(
    preprocess_manifest["artifacts"].values(),
    key=lambda profile: profile["timestamp_range"]["start_utc"],
)
feature_profiles = sorted(
    feature_manifest["artifacts"].values(),
    key=lambda profile: profile["timestamp_range"]["start_utc"],
)
source_parts = [pd.read_parquet(profile["path"]) for profile in source_profiles]
feature_parts = [pd.read_parquet(profile["path"]) for profile in feature_profiles]

# No source-partition marker is retained in either analysis frame.
processed_joined = (
    pd.concat(source_parts, ignore_index=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)
features_joined = (
    pd.concat(feature_parts, ignore_index=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)
seam_timestamp = pd.Timestamp(source_profiles[1]["timestamp_range"]["start_utc"])

# This station's own view, extracted from the joined multi-station dataset.
processed = extract_station_frame(processed_joined, STATION_ID)
features = extract_station_frame(features_joined, STATION_ID)
predictor_columns = list(feature_column_names(DEFAULT_FEATURE_CONFIG))
target_columns = list(target_column_names(DEFAULT_FEATURE_CONFIG))
source_columns = list(processed.columns)
horizon_hours = int(feature_manifest["configuration"]["horizon_hours"])

In [ ]:
validation_records = []


def record_check(check: str, detail: str) -> None:
    """Append a successful validation result for compact reporting."""
    validation_records.append({"check": check, "status": "PASS", "detail": detail})


assert preprocess_manifest["target_station_id"] == STATION_ID
assert STATION_ID in preprocess_manifest["station_ids"]
assert STATION_ID in feature_manifest["station_ids"]
assert preprocess_manifest["rows"]["total"] == len(processed_joined)
record_check("Manifest identity", "Both joined manifests cover the selected station")

station_rows = station_catalog.loc[station_catalog["commonid"].eq(STATION_ID)]
assert len(station_rows) == 1
station = station_rows.iloc[0]
alarm_threshold_cm = float(station["defaultAlarmValueCm"])
assert np.isfinite(alarm_threshold_cm)
record_check(
    "Station metadata",
    f"Unique catalog row; alarm threshold {alarm_threshold_cm:.0f} cm",
)

assert list(raw_water.columns) == ["value", "sourceDate", "station_id"]
assert list(raw_weather.columns) == [
    "time",
    "precipitation",
    "temperature_2m",
    "station_id",
    "requested_latitude",
    "requested_longitude",
    "grid_latitude",
    "grid_longitude",
    "weather_model",
]
assert raw_water["station_id"].drop_duplicates().tolist() == [STATION_ID]
assert raw_weather["station_id"].drop_duplicates().tolist() == [STATION_ID]
record_check("Raw schemas", "Gauge and INCA columns match the expected contracts")

for profile, frame in zip(source_profiles, source_parts, strict=True):
    validate_profile(profile, frame)
for profile, frame in zip(feature_profiles, feature_parts, strict=True):
    validate_profile(profile, frame)
for manifest in (preprocess_manifest, feature_manifest):
    generator = manifest["generator"]
    assert sha256(Path(generator["module"])) == generator["sha256"]
record_check(
    "Hashes and profiles",
    "All artifact and generator hashes, schemas, rows, ranges, and nulls match",
)

assert_utc_ordered_unique(raw_water, "sourceDate", hourly=False)
assert_utc_ordered_unique(raw_weather, "time", hourly=True)
assert_utc_ordered_unique(processed, "timestamp", hourly=True)
assert_utc_ordered_unique(features, "timestamp", hourly=True)
record_check(
    "UTC timeline",
    "Raw timestamps are ordered/unique; processed and feature grids are hourly",
)

assert (
    source_parts[0]["timestamp"].iloc[-1] + pd.Timedelta(hours=1)
    == (source_parts[1]["timestamp"].iloc[0])
)
assert (
    feature_parts[0]["timestamp"].iloc[-1] + pd.Timedelta(hours=1)
    == (feature_parts[1]["timestamp"].iloc[0])
)
assert processed["timestamp"].equals(features["timestamp"])
assert processed["station_id"].drop_duplicates().tolist() == [STATION_ID]
record_check(
    "Concatenation and alignment",
    f"Single chronological history; {len(processed):,} aligned hourly rows",
)

assert list(features.columns[: len(source_columns)]) == source_columns
for column in source_columns:
    assert features[column].equals(processed[column])
assert set(predictor_columns).issubset(features.columns)
assert set(target_columns).issubset(features.columns)
assert len(target_columns) == horizon_hours == 24
record_check(
    "Feature contract",
    "All source columns are byte-logically preserved; predictor/target lists exist",
)

station_source_parts = [
    extract_station_frame(part, STATION_ID) for part in source_parts
]
station_feature_parts = [
    extract_station_frame(part, STATION_ID) for part in feature_parts
]
for source_part, feature_part in zip(
    station_source_parts, station_feature_parts, strict=True
):
    expected_valid = target_eligibility(source_part, horizon_hours)
    assert feature_part["target_valid"].astype(bool).equals(expected_valid)
    for offset, target in enumerate(target_columns, start=1):
        expected = source_part["water_level"].shift(-offset).where(expected_valid)
        assert feature_part[target].equals(expected)
record_check(
    "Target contract",
    "Every target equals water level at t+h and is blank unless the full vector is observed",
)

raw_water_by_time = raw_water.set_index("sourceDate")["value"]
raw_at_processed = processed["timestamp"].map(raw_water_by_time)
raw_observed = raw_at_processed.notna()
assert np.allclose(
    processed.loc[raw_observed, "water_level"],
    raw_at_processed.loc[raw_observed],
    equal_nan=True,
)
assert not processed.loc[raw_observed, "imputed"].any()

weather_by_time = raw_weather.set_index("time")
for column in ("precipitation", "temperature_2m"):
    expected = processed["timestamp"].map(weather_by_time[column])
    assert np.allclose(processed[column], expected, equal_nan=True)
assert processed["timestamp"].iloc[0] == raw_weather["time"].iloc[0]
assert processed["timestamp"].iloc[-1] == raw_weather["time"].iloc[-1]
record_check(
    "Raw-to-processed coverage",
    "All raw gauge values and selected INCA variables align without alteration",
)

display(pd.DataFrame(validation_records))

In [ ]:
inventory = pd.DataFrame(
    [
        {
            "role": "Station catalog",
            "rows": len(station_catalog),
            "start_utc": pd.NaT,
            "end_utc": pd.NaT,
            "sha256": sha256(PATHS["station_catalog"]),
        },
        {
            "role": "Raw water level",
            "rows": len(raw_water),
            "start_utc": raw_water["sourceDate"].min(),
            "end_utc": raw_water["sourceDate"].max(),
            "sha256": sha256(PATHS["raw_water"]),
        },
        {
            "role": "Raw INCA weather",
            "rows": len(raw_weather),
            "start_utc": raw_weather["time"].min(),
            "end_utc": raw_weather["time"].max(),
            "sha256": sha256(PATHS["raw_weather"]),
        },
        {
            "role": "Preprocessed full history",
            "rows": len(processed),
            "start_utc": processed["timestamp"].min(),
            "end_utc": processed["timestamp"].max(),
            "sha256": "Manifest-validated components",
        },
        {
            "role": "Feature full history",
            "rows": len(features),
            "start_utc": features["timestamp"].min(),
            "end_utc": features["timestamp"].max(),
            "sha256": "Manifest-validated components",
        },
    ]
)
station_summary = pd.DataFrame(
    {
        "field": [
            "Station",
            "Water body",
            "Region",
            "Coordinates",
            "Altitude",
            "Alarm threshold",
        ],
        "value": [
            f"{station['stationName']} ({STATION_ID})",
            station["water"],
            station["region"],
            f"{station['latitude']:.5f}, {station['longitude']:.5f}",
            f"{station['altitudeM']:.2f} m",
            f"{alarm_threshold_cm:.0f} cm",
        ],
    }
)
display(station_summary, inventory)

## 3. Data-quality audit

Gauge gaps are reconstructed on the same hourly grid as the processed data.
Consecutive missing runs of at most six hours should be interpolated and flagged;
longer runs should remain null. Weather anomalies are reported, never clipped.
Monthly completeness uses all in-scope hours, including partial first and last
months.

In [ ]:
def run_table(mask: pd.Series, timestamps: pd.Series) -> pd.DataFrame:
    """Return start, end, and duration for consecutive true runs."""
    mask = mask.fillna(False).astype(bool).reset_index(drop=True)
    timestamps = timestamps.reset_index(drop=True)
    run_id = mask.ne(mask.shift(fill_value=False)).cumsum()
    rows = []
    for index in mask[mask].groupby(run_id[mask]).groups.values():
        positions = np.asarray(list(index), dtype=int)
        rows.append(
            {
                "start_utc": timestamps.iloc[positions[0]],
                "end_utc": timestamps.iloc[positions[-1]],
                "hours": len(positions),
            }
        )
    return pd.DataFrame(rows, columns=["start_utc", "end_utc", "hours"])


raw_grid_missing = raw_at_processed.isna()
raw_gap_runs = run_table(raw_grid_missing, processed["timestamp"])
remaining_gap_runs = run_table(processed["water_level"].isna(), processed["timestamp"])
imputed_runs = run_table(processed["imputed"], processed["timestamp"])

# Confirm the stored interpolation mask exactly matches the six-hour rule.
raw_gap_id = raw_grid_missing.ne(raw_grid_missing.shift()).cumsum()
raw_gap_length = raw_grid_missing.groupby(raw_gap_id).transform("sum")
expected_imputed = raw_grid_missing & raw_gap_length.le(MAX_INTERPOLATION_GAP_HOURS)
assert processed["imputed"].equals(expected_imputed.rename("imputed"))
assert processed.loc[processed["imputed"], "water_level"].notna().all()
assert (
    processed.loc[raw_grid_missing & ~processed["imputed"], "water_level"].isna().all()
)

quality_overview = pd.DataFrame(
    [
        {
            "dataset": "Raw gauge",
            "rows": len(raw_water),
            "duplicate_timestamps": raw_water["sourceDate"].duplicated().sum(),
            "missing_water": int(raw_grid_missing.sum()),
            "missing_precipitation": np.nan,
            "missing_temperature": np.nan,
        },
        {
            "dataset": "Raw INCA",
            "rows": len(raw_weather),
            "duplicate_timestamps": raw_weather["time"].duplicated().sum(),
            "missing_water": np.nan,
            "missing_precipitation": raw_weather["precipitation"].isna().sum(),
            "missing_temperature": raw_weather["temperature_2m"].isna().sum(),
        },
        {
            "dataset": "Processed full history",
            "rows": len(processed),
            "duplicate_timestamps": processed["timestamp"].duplicated().sum(),
            "missing_water": processed["water_level"].isna().sum(),
            "missing_precipitation": processed["precipitation"].isna().sum(),
            "missing_temperature": processed["temperature_2m"].isna().sum(),
        },
        {
            "dataset": "Feature full history",
            "rows": len(features),
            "duplicate_timestamps": features["timestamp"].duplicated().sum(),
            "missing_water": features["water_level"].isna().sum(),
            "missing_precipitation": features["precipitation"].isna().sum(),
            "missing_temperature": features["temperature_2m"].isna().sum(),
        },
    ]
)

interpolation_summary = pd.DataFrame(
    {
        "metric": [
            "Raw missing hourly observations",
            "Raw missing-gap episodes",
            "Interpolated observations",
            "Interpolated episodes",
            "Maximum interpolated run (hours)",
            "Remaining missing observations",
            "Remaining long-gap episodes",
            "Maximum remaining gap (hours)",
        ],
        "value": [
            int(raw_grid_missing.sum()),
            len(raw_gap_runs),
            int(processed["imputed"].sum()),
            len(imputed_runs),
            int(imputed_runs["hours"].max()),
            int(processed["water_level"].isna().sum()),
            len(remaining_gap_runs),
            int(remaining_gap_runs["hours"].max()),
        ],
    }
)

gap_distribution = (
    raw_gap_runs["hours"]
    .value_counts()
    .rename("raw_gap_episodes")
    .to_frame()
    .join(
        remaining_gap_runs["hours"].value_counts().rename("remaining_gap_episodes"),
        how="outer",
    )
    .fillna(0)
    .astype(int)
    .sort_index()
    .rename_axis("gap_length_hours")
    .reset_index()
)

negative_precipitation = processed.loc[
    processed["precipitation"].lt(0),
    ["timestamp", "precipitation"],
]
weather_anomalies = pd.DataFrame(
    {
        "metric": [
            "Missing precipitation values",
            "Missing temperature values",
            "Negative precipitation values",
            "Minimum precipitation (mm)",
            "Maximum precipitation (mm)",
        ],
        "value": [
            int(processed["precipitation"].isna().sum()),
            int(processed["temperature_2m"].isna().sum()),
            len(negative_precipitation),
            processed["precipitation"].min(),
            processed["precipitation"].max(),
        ],
    }
)
display(
    quality_overview,
    interpolation_summary,
    gap_distribution,
    weather_anomalies,
    negative_precipitation,
)

In [ ]:
month_key = processed["timestamp"].dt.tz_localize(None).dt.to_period("M")
monthly_completeness = (
    processed.assign(month=month_key)
    .groupby("month", observed=True)
    .agg(
        in_scope_hours=("timestamp", "size"),
        observed_water_hours=("water_level", "count"),
        imputed_water_hours=("imputed", "sum"),
        precipitation_hours=("precipitation", "count"),
        temperature_hours=("temperature_2m", "count"),
    )
    .reset_index()
)
monthly_completeness["month"] = (
    monthly_completeness["month"].dt.to_timestamp().dt.tz_localize("UTC")
)
for output, numerator in {
    "water_complete_pct": "observed_water_hours",
    "precipitation_complete_pct": "precipitation_hours",
    "temperature_complete_pct": "temperature_hours",
}.items():
    monthly_completeness[output] = (
        100 * monthly_completeness[numerator] / monthly_completeness["in_scope_hours"]
    )

worst_months = monthly_completeness.nsmallest(12, "water_complete_pct")
display(worst_months)

coverage_figure = go.Figure()
for column, label, color in [
    ("water_complete_pct", "Water level", PLOT_COLORS["water"]),
    ("precipitation_complete_pct", "Precipitation", PLOT_COLORS["rain"]),
    ("temperature_complete_pct", "Temperature", PLOT_COLORS["temperature"]),
]:
    coverage_figure.add_trace(
        go.Scatter(
            x=monthly_completeness["month"],
            y=monthly_completeness[column],
            mode="lines",
            name=label,
            line={"color": color},
        )
    )
coverage_figure.update_layout(
    title="Monthly observation completeness across the full history",
    template=PLOT_TEMPLATE,
    height=430,
    yaxis_title="Complete in-scope hours (%)",
    yaxis_range=[0, 101],
    hovermode="x unified",
)
coverage_figure.show()

## 4. Full-history hydrometeorological narrative

Decade-scale views are aggregated to daily, annual, monthly, or hourly summaries.
This keeps the interactive outputs responsive while preserving the trends relevant
to the thesis. Hourly rows are retained only later for focused event windows.

In [ ]:
daily = (
    processed.set_index("timestamp")
    .resample("D")
    .agg(
        water_min=("water_level", "min"),
        water_mean=("water_level", "mean"),
        water_max=("water_level", "max"),
        rainfall_total=("precipitation", lambda values: values.sum(min_count=1)),
        temperature_mean=("temperature_2m", "mean"),
    )
    .reset_index()
)

overview_figure = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "Daily water-level range",
        "Daily rainfall total",
        "Daily mean temperature",
    ),
)
overview_figure.add_trace(
    go.Scatter(
        x=daily["timestamp"],
        y=daily["water_max"],
        mode="lines",
        line={"width": 0},
        showlegend=False,
        hoverinfo="skip",
    ),
    row=1,
    col=1,
)
overview_figure.add_trace(
    go.Scatter(
        x=daily["timestamp"],
        y=daily["water_min"],
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(23, 107, 135, 0.22)",
        line={"color": PLOT_COLORS["water"], "width": 0.8},
        name="Daily min–max",
    ),
    row=1,
    col=1,
)
overview_figure.add_trace(
    go.Scatter(
        x=daily["timestamp"],
        y=daily["water_mean"],
        mode="lines",
        line={"color": PLOT_COLORS["water"], "width": 1},
        name="Daily mean",
    ),
    row=1,
    col=1,
)
overview_figure.add_hline(
    y=alarm_threshold_cm,
    line_dash="dash",
    line_color=PLOT_COLORS["threshold"],
    annotation_text=f"Alarm threshold: {alarm_threshold_cm:.0f} cm",
    row=1,
    col=1,
)
overview_figure.add_trace(
    go.Bar(
        x=daily["timestamp"],
        y=daily["rainfall_total"],
        marker_color=PLOT_COLORS["rain"],
        name="Rainfall",
    ),
    row=2,
    col=1,
)
overview_figure.add_trace(
    go.Scatter(
        x=daily["timestamp"],
        y=daily["temperature_mean"],
        mode="lines",
        line={"color": PLOT_COLORS["temperature"], "width": 1},
        name="Temperature",
    ),
    row=3,
    col=1,
)
overview_figure.update_yaxes(title_text="Water level (cm)", row=1, col=1)
overview_figure.update_yaxes(title_text="Rain (mm)", row=2, col=1)
overview_figure.update_yaxes(title_text="Temperature (°C)", row=3, col=1)
overview_figure.update_layout(
    title="Full-history daily hydro-meteorological overview",
    template=PLOT_TEMPLATE,
    height=900,
    hovermode="x unified",
    legend={"orientation": "h", "y": 1.04},
)
overview_figure.show()

In [ ]:
def binned_distribution(
    values: pd.Series,
    *,
    bins: int = 60,
    upper_quantile: float | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Return compact histogram midpoints and counts for finite values."""
    clean = pd.to_numeric(values, errors="coerce").dropna().to_numpy()
    clean = clean[np.isfinite(clean)]
    if upper_quantile is not None:
        clean = clean[clean <= np.quantile(clean, upper_quantile)]
    counts, edges = np.histogram(clean, bins=bins)
    return (edges[:-1] + edges[1:]) / 2, counts


distribution_figure = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Water level",
        "Hourly precipitation (through 99.5th percentile)",
        "Air temperature",
    ),
)
distribution_specs = [
    (processed["water_level"], "Water level", PLOT_COLORS["water"], None),
    (
        processed["precipitation"],
        "Precipitation",
        PLOT_COLORS["rain"],
        0.995,
    ),
    (
        processed["temperature_2m"],
        "Temperature",
        PLOT_COLORS["temperature"],
        None,
    ),
]
for column_index, (values, label, color, quantile) in enumerate(
    distribution_specs,
    start=1,
):
    midpoint, count = binned_distribution(
        values,
        upper_quantile=quantile,
    )
    distribution_figure.add_trace(
        go.Bar(x=midpoint, y=count, name=label, marker_color=color),
        row=1,
        col=column_index,
    )
distribution_figure.update_xaxes(title_text="cm", row=1, col=1)
distribution_figure.update_xaxes(title_text="mm/h", row=1, col=2)
distribution_figure.update_xaxes(title_text="°C", row=1, col=3)
distribution_figure.update_yaxes(title_text="Hourly observations", row=1, col=1)
distribution_figure.update_layout(
    title="Core full-history distributions (pre-binned for compact output)",
    template=PLOT_TEMPLATE,
    height=430,
    showlegend=False,
)
distribution_figure.show()

In [ ]:
annual = (
    processed.assign(year=processed["timestamp"].dt.year)
    .groupby("year")
    .agg(
        mean_water_level=("water_level", "mean"),
        maximum_water_level=("water_level", "max"),
        total_rainfall=("precipitation", lambda values: values.sum(min_count=1)),
        mean_temperature=("temperature_2m", "mean"),
        water_coverage=("water_level", "count"),
        hours=("timestamp", "size"),
    )
    .reset_index()
)
annual["water_coverage_pct"] = 100 * annual["water_coverage"] / annual["hours"]

annual_figure = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Mean and maximum water level",
        "Total precipitation",
        "Mean temperature",
    ),
)
annual_figure.add_trace(
    go.Scatter(
        x=annual["year"],
        y=annual["mean_water_level"],
        mode="lines+markers",
        name="Mean water",
        line={"color": PLOT_COLORS["water"]},
    ),
    row=1,
    col=1,
)
annual_figure.add_trace(
    go.Scatter(
        x=annual["year"],
        y=annual["maximum_water_level"],
        mode="lines+markers",
        name="Maximum water",
        line={"color": "#0B3C5D", "dash": "dot"},
    ),
    row=1,
    col=1,
)
annual_figure.add_trace(
    go.Bar(
        x=annual["year"],
        y=annual["total_rainfall"],
        name="Rainfall",
        marker_color=PLOT_COLORS["rain"],
    ),
    row=1,
    col=2,
)
annual_figure.add_trace(
    go.Scatter(
        x=annual["year"],
        y=annual["mean_temperature"],
        mode="lines+markers",
        name="Temperature",
        line={"color": PLOT_COLORS["temperature"]},
    ),
    row=1,
    col=3,
)
annual_figure.update_yaxes(title_text="cm", row=1, col=1)
annual_figure.update_yaxes(title_text="mm", row=1, col=2)
annual_figure.update_yaxes(title_text="°C", row=1, col=3)
annual_figure.update_layout(
    title="Annual trends (partial boundary years are retained)",
    template=PLOT_TEMPLATE,
    height=430,
    legend={"orientation": "h", "y": 1.12},
)
annual_figure.show()
display(annual)

In [ ]:
monthly_seasonality = (
    processed.assign(month=processed["timestamp"].dt.month)
    .groupby("month")
    .agg(
        water_level=("water_level", "mean"),
        precipitation=("precipitation", "mean"),
        temperature=("temperature_2m", "mean"),
    )
    .reset_index()
)
hourly_seasonality = (
    processed.assign(hour=processed["timestamp"].dt.hour)
    .groupby("hour")
    .agg(
        water_level=("water_level", "mean"),
        precipitation=("precipitation", "mean"),
        temperature=("temperature_2m", "mean"),
    )
    .reset_index()
)

seasonality_figure = make_subplots(
    rows=2,
    cols=1,
    specs=[[{"secondary_y": True}], [{"secondary_y": True}]],
    subplot_titles=("Monthly climatology", "UTC hourly climatology"),
    vertical_spacing=0.16,
)
for row, frame, x_column in [
    (1, monthly_seasonality, "month"),
    (2, hourly_seasonality, "hour"),
]:
    seasonality_figure.add_trace(
        go.Scatter(
            x=frame[x_column],
            y=frame["water_level"],
            mode="lines+markers",
            name=f"Water level ({x_column})",
            line={"color": PLOT_COLORS["water"]},
        ),
        row=row,
        col=1,
        secondary_y=False,
    )
    seasonality_figure.add_trace(
        go.Scatter(
            x=frame[x_column],
            y=frame["temperature"],
            mode="lines+markers",
            name=f"Temperature ({x_column})",
            line={"color": PLOT_COLORS["temperature"]},
        ),
        row=row,
        col=1,
        secondary_y=True,
    )
    seasonality_figure.add_trace(
        go.Bar(
            x=frame[x_column],
            y=frame["precipitation"],
            name=f"Precipitation ({x_column})",
            marker_color=PLOT_COLORS["rain"],
            opacity=0.35,
        ),
        row=row,
        col=1,
        secondary_y=True,
    )
    seasonality_figure.update_yaxes(
        title_text="Water level (cm)", row=row, col=1, secondary_y=False
    )
    seasonality_figure.update_yaxes(
        title_text="°C / mean mm/h", row=row, col=1, secondary_y=True
    )
seasonality_figure.update_xaxes(
    title_text="Calendar month (UTC)", dtick=1, row=1, col=1
)
seasonality_figure.update_xaxes(title_text="Hour of day (UTC)", dtick=2, row=2, col=1)
seasonality_figure.update_layout(
    title="Monthly and diurnal seasonality",
    template=PLOT_TEMPLATE,
    height=760,
    barmode="overlay",
    legend={"orientation": "h", "y": 1.09},
)
seasonality_figure.show()

In [ ]:
monthly_coverage = (
    processed.assign(month=month_key)
    .groupby("month", observed=True)
    .agg(
        raw_observed=("water_level", lambda values: 0),
        imputed=("imputed", "sum"),
        processed_observed=("water_level", "count"),
        total=("timestamp", "size"),
    )
    .reset_index()
)
raw_month_counts = (
    pd.DataFrame({"month": month_key, "raw_observed": raw_observed.astype(int)})
    .groupby("month", observed=True)["raw_observed"]
    .sum()
)
monthly_coverage["raw_observed"] = monthly_coverage["month"].map(raw_month_counts)
monthly_coverage["remaining_missing"] = (
    monthly_coverage["total"] - monthly_coverage["processed_observed"]
)
monthly_coverage["month"] = (
    monthly_coverage["month"].dt.to_timestamp().dt.tz_localize("UTC")
)

raw_processed_figure = go.Figure()
raw_processed_figure.add_trace(
    go.Bar(
        x=monthly_coverage["month"],
        y=monthly_coverage["raw_observed"],
        name="Raw observed",
        marker_color=PLOT_COLORS["water"],
    )
)
raw_processed_figure.add_trace(
    go.Bar(
        x=monthly_coverage["month"],
        y=monthly_coverage["imputed"],
        name="Interpolated (≤6 h gaps)",
        marker_color=PLOT_COLORS["imputed"],
    )
)
raw_processed_figure.add_trace(
    go.Bar(
        x=monthly_coverage["month"],
        y=monthly_coverage["remaining_missing"],
        name="Remaining missing (>6 h gaps)",
        marker_color=PLOT_COLORS["missing"],
    )
)
raw_processed_figure.update_layout(
    title="Raw versus processed water-level coverage by month",
    template=PLOT_TEMPLATE,
    height=460,
    barmode="stack",
    yaxis_title="Hourly positions",
    hovermode="x unified",
    legend={"orientation": "h", "y": 1.08},
)
raw_processed_figure.show()

## 5. Alarm-threshold exceedances and event context

The threshold is read directly from the station catalog, not hard-coded. An episode
is a consecutive run of hourly water levels at or above the threshold. The largest
episodes are ranked by peak level; rainfall totals use the 24 and 72 hours strictly
preceding each episode start. Hourly context is limited to ±7 days around the five
largest episodes.

In [ ]:
exceedance_mask = processed["water_level"].ge(alarm_threshold_cm)
episode_runs = run_table(exceedance_mask, processed["timestamp"])
episode_rows = []
for episode_number, episode in episode_runs.iterrows():
    in_episode = processed["timestamp"].between(
        episode["start_utc"], episode["end_utc"]
    )
    episode_data = processed.loc[in_episode]
    peak_index = episode_data["water_level"].idxmax()
    start = episode["start_utc"]
    preceding_24 = processed["timestamp"].between(
        start - pd.Timedelta(hours=24),
        start,
        inclusive="left",
    )
    preceding_72 = processed["timestamp"].between(
        start - pd.Timedelta(hours=72),
        start,
        inclusive="left",
    )
    episode_rows.append(
        {
            "episode": int(episode_number + 1),
            "start_utc": start,
            "end_utc": episode["end_utc"],
            "duration_hours": int(episode["hours"]),
            "peak_utc": processed.loc[peak_index, "timestamp"],
            "peak_cm": processed.loc[peak_index, "water_level"],
            "excess_above_threshold_cm": (
                processed.loc[peak_index, "water_level"] - alarm_threshold_cm
            ),
            "preceding_rain_24h_mm": processed.loc[preceding_24, "precipitation"].sum(
                min_count=1
            ),
            "preceding_rain_72h_mm": processed.loc[preceding_72, "precipitation"].sum(
                min_count=1
            ),
            "imputed_hours": int(episode_data["imputed"].sum()),
        }
    )
episodes = pd.DataFrame(episode_rows).sort_values(
    ["peak_cm", "duration_hours"], ascending=[False, False]
)
largest_events = episodes.head(TOP_EVENT_COUNT).reset_index(drop=True)
assert not episodes.empty
display(episodes.reset_index(drop=True), largest_events)

In [ ]:
event_figure = make_subplots(
    rows=TOP_EVENT_COUNT,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.055,
    specs=[[{"secondary_y": True}]] * TOP_EVENT_COUNT,
    subplot_titles=[
        (f"Peak {event.peak_cm:.0f} cm at {event.peak_utc:%Y-%m-%d %H:%M UTC}")
        for event in largest_events.itertuples()
    ],
)
for row_number, event in enumerate(largest_events.itertuples(), start=1):
    window = processed["timestamp"].between(
        event.peak_utc - pd.Timedelta(days=EVENT_CONTEXT_DAYS),
        event.peak_utc + pd.Timedelta(days=EVENT_CONTEXT_DAYS),
    )
    context = processed.loc[window]
    event_figure.add_trace(
        go.Bar(
            x=context["timestamp"],
            y=context["precipitation"],
            name="Hourly precipitation",
            marker_color=PLOT_COLORS["rain"],
            opacity=0.35,
            showlegend=row_number == 1,
        ),
        row=row_number,
        col=1,
        secondary_y=True,
    )
    event_figure.add_trace(
        go.Scatter(
            x=context["timestamp"],
            y=context["water_level"],
            mode="lines",
            name="Water level",
            line={"color": PLOT_COLORS["water"], "width": 1.6},
            showlegend=row_number == 1,
        ),
        row=row_number,
        col=1,
        secondary_y=False,
    )
    event_figure.add_hline(
        y=alarm_threshold_cm,
        line_dash="dash",
        line_color=PLOT_COLORS["threshold"],
        row=row_number,
        col=1,
    )
    event_figure.update_yaxes(title_text="cm", row=row_number, col=1, secondary_y=False)
    event_figure.update_yaxes(
        title_text="mm/h", row=row_number, col=1, secondary_y=True
    )
event_figure.update_layout(
    title="Largest alarm-threshold episodes: ±7-day water and rainfall context",
    template=PLOT_TEMPLATE,
    height=1_280,
    hovermode="x unified",
    legend={"orientation": "h", "y": 1.025},
)
event_figure.show()

## 6. Time-series structure and stationarity

Stationarity tests use the longest consecutive run of **observed** water levels:
interpolated and missing hours are excluded before the run is selected. ADF tests
the null of a unit root; KPSS tests the null of level stationarity. Applying both to
levels and first differences avoids relying on one test alone. ACF and PACF are
shown through one week (168 hourly lags).

In [ ]:
observed_runs = run_table(
    processed["water_level"].notna() & ~processed["imputed"],
    processed["timestamp"],
)
longest_observed_run = observed_runs.loc[observed_runs["hours"].idxmax()]
longest_mask = processed["timestamp"].between(
    longest_observed_run["start_utc"],
    longest_observed_run["end_utc"],
)
observed_segment = processed.loc[longest_mask, "water_level"].astype(float)
assert observed_segment.notna().all()
first_difference = observed_segment.diff().dropna()


def stationarity_tests(values: pd.Series, transformation: str) -> list[dict]:
    """Run ADF and KPSS and return compact, interpretable test records."""
    records = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        adf_result = adfuller(values, autolag="AIC")
        kpss_result = kpss(values, regression="c", nlags="auto")
    records.append(
        {
            "transformation": transformation,
            "test": "ADF (H0: unit root)",
            "statistic": adf_result[0],
            "p_value": adf_result[1],
            "lags": int(adf_result[2]),
            "observations": int(adf_result[3]),
            "decision_at_5pct": (
                "Reject unit root" if adf_result[1] < 0.05 else "Do not reject"
            ),
        }
    )
    records.append(
        {
            "transformation": transformation,
            "test": "KPSS (H0: stationary)",
            "statistic": kpss_result[0],
            "p_value": kpss_result[1],
            "lags": int(kpss_result[2]),
            "observations": len(values),
            "decision_at_5pct": (
                "Reject stationarity" if kpss_result[1] < 0.05 else "Do not reject"
            ),
        }
    )
    return records


stationarity_results = pd.DataFrame(
    stationarity_tests(observed_segment, "Level")
    + stationarity_tests(first_difference, "First difference")
)
segment_summary = pd.DataFrame(
    {
        "start_utc": [longest_observed_run["start_utc"]],
        "end_utc": [longest_observed_run["end_utc"]],
        "observed_hours": [int(longest_observed_run["hours"])],
    }
)
display(segment_summary, stationarity_results)

In [ ]:
maximum_lag = 168
acf_values, acf_confidence = acf(
    observed_segment,
    nlags=maximum_lag,
    alpha=0.05,
    fft=True,
)
pacf_values, pacf_confidence = pacf(
    observed_segment,
    nlags=maximum_lag,
    alpha=0.05,
    method="ywm",
)
lags = np.arange(maximum_lag + 1)

correlation_figure = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=("Autocorrelation", "Partial autocorrelation"),
    vertical_spacing=0.12,
)
for row, values, confidence, label, color in [
    (1, acf_values, acf_confidence, "ACF", PLOT_COLORS["water"]),
    (2, pacf_values, pacf_confidence, "PACF", "#7A5195"),
]:
    lower = confidence[:, 0] - values
    upper = confidence[:, 1] - values
    correlation_figure.add_trace(
        go.Bar(x=lags, y=values, name=label, marker_color=color),
        row=row,
        col=1,
    )
    correlation_figure.add_trace(
        go.Scatter(
            x=lags,
            y=upper,
            mode="lines",
            line={"width": 0},
            showlegend=False,
            hoverinfo="skip",
        ),
        row=row,
        col=1,
    )
    correlation_figure.add_trace(
        go.Scatter(
            x=lags,
            y=lower,
            mode="lines",
            fill="tonexty",
            fillcolor="rgba(108, 117, 125, 0.18)",
            line={"width": 0},
            showlegend=False,
            hoverinfo="skip",
        ),
        row=row,
        col=1,
    )
    correlation_figure.add_hline(y=0, line_color="#333", row=row, col=1)
    correlation_figure.update_yaxes(title_text=label, row=row, col=1)
correlation_figure.update_xaxes(title_text="Hourly lag", dtick=24, row=2, col=1)
correlation_figure.update_layout(
    title="ACF and PACF through one week of hourly lags",
    template=PLOT_TEMPLATE,
    height=700,
    showlegend=False,
)
correlation_figure.show()

## 7. Weather relationships with future water-level changes

Spearman correlations capture monotonic, not necessarily linear, relationships.
Stored trailing precipitation sums and temperature means at 6, 24, 72, and 168
hours are related to future water-level changes at 1, 6, 12, and 24 hours. Rows are
used pairwise only when the aggregate and both water-level endpoints are finite.
These are descriptive full-history associations, not causal or holdout estimates.

In [ ]:
relationship_records = []
for weather_variable, statistic in [
    ("precipitation", "rolling_sum"),
    ("temperature_2m", "rolling_mean"),
]:
    for window in RELATIONSHIP_WINDOWS:
        aggregate = f"{weather_variable}_{statistic}_{window}h"
        for horizon in RELATIONSHIP_HORIZONS:
            future_change = (
                features["water_level"].shift(-horizon) - features["water_level"]
            )
            pair = pd.concat(
                [features[aggregate], future_change.rename("future_change")],
                axis=1,
            ).dropna()
            relationship_records.append(
                {
                    "weather_aggregate": aggregate,
                    "horizon_hours": horizon,
                    "spearman_rho": pair[aggregate].corr(
                        pair["future_change"], method="spearman"
                    ),
                    "paired_rows": len(pair),
                }
            )
weather_relationships = pd.DataFrame(relationship_records)
relationship_matrix = weather_relationships.pivot(
    index="weather_aggregate",
    columns="horizon_hours",
    values="spearman_rho",
)
strongest_relationships = (
    weather_relationships.assign(
        absolute_rho=weather_relationships["spearman_rho"].abs()
    )
    .sort_values("absolute_rho", ascending=False)
    .head(12)
)
display(relationship_matrix, strongest_relationships)

relationship_figure = go.Figure(
    go.Heatmap(
        z=relationship_matrix.to_numpy(),
        x=[f"t+{value} h" for value in relationship_matrix.columns],
        y=relationship_matrix.index,
        colorscale="RdBu",
        zmid=0,
        zmin=-1,
        zmax=1,
        colorbar={"title": "Spearman ρ"},
        hovertemplate="%{y}<br>%{x}<br>ρ=%{z:.3f}<extra></extra>",
    )
)
relationship_figure.update_layout(
    title="Weather aggregates versus future water-level change",
    template=PLOT_TEMPLATE,
    height=520,
    xaxis_title="Forecast horizon",
)
relationship_figure.show()

## 8. Predictor and target diagnostics

Health summaries cover every manifest-declared predictor and target. Constant and
infinite-value checks are separate from missingness because they imply different
modeling remedies. The complete predictor correlation matrix is retained as an
interactive heatmap; highly redundant pairs are listed when |r| > 0.95. Target
readiness is summarized per horizon, followed by compact pre-binned distributions
of target change relative to issue time.

In [ ]:
def column_health(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Summarize dtype, finite coverage, nulls, infinities, and constants."""
    records = []
    for column in columns:
        series = frame[column]
        numeric = pd.to_numeric(series, errors="coerce")
        finite = np.isfinite(numeric.dropna().to_numpy(dtype=float))
        records.append(
            {
                "column": column,
                "dtype": str(series.dtype),
                "non_null": int(series.notna().sum()),
                "missing": int(series.isna().sum()),
                "missing_pct": 100 * series.isna().mean(),
                "infinite": int((~finite).sum()),
                "constant_non_null": bool(series.dropna().nunique() <= 1),
                "minimum": numeric.min(),
                "maximum": numeric.max(),
            }
        )
    return pd.DataFrame(records)


predictor_health = column_health(features, predictor_columns)
target_health = column_health(features, target_columns)
assert predictor_health["infinite"].sum() == 0
assert target_health["infinite"].sum() == 0
display(predictor_health, target_health)

In [ ]:
predictor_numeric = features[predictor_columns].astype(float)
predictor_correlation = predictor_numeric.corr(method="pearson")
upper_triangle = np.triu(np.ones(predictor_correlation.shape, dtype=bool), k=1)
high_correlation_pairs = (
    predictor_correlation.where(upper_triangle)
    .stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={"level_0": "predictor_a", "level_1": "predictor_b"})
)
high_correlation_pairs = (
    high_correlation_pairs.loc[high_correlation_pairs["correlation"].abs().gt(0.95)]
    .assign(absolute_correlation=lambda frame: frame["correlation"].abs())
    .sort_values("absolute_correlation", ascending=False)
    .reset_index(drop=True)
)
display(high_correlation_pairs)

predictor_correlation_figure = go.Figure(
    go.Heatmap(
        z=predictor_correlation.to_numpy(),
        x=predictor_correlation.columns,
        y=predictor_correlation.index,
        colorscale="RdBu",
        zmid=0,
        zmin=-1,
        zmax=1,
        colorbar={"title": "Pearson r"},
        hovertemplate="%{y}<br>%{x}<br>r=%{z:.3f}<extra></extra>",
    )
)
predictor_correlation_figure.update_layout(
    title="Full predictor correlation matrix",
    template=PLOT_TEMPLATE,
    height=940,
    width=1_050,
    xaxis={"tickangle": -55},
)
predictor_correlation_figure.show()

In [ ]:
target_coverage = pd.DataFrame(
    {
        "horizon_hours": range(1, horizon_hours + 1),
        "target": target_columns,
        "valid_rows": [features[column].notna().sum() for column in target_columns],
        "valid_pct": [
            100 * features[column].notna().mean() for column in target_columns
        ],
    }
)
target_coverage["all_target_vector_valid_pct"] = 100 * features["target_valid"].mean()
display(target_coverage)

target_change_figure = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[f"t+{horizon} h" for horizon in RELATIONSHIP_HORIZONS],
)
for position, horizon in enumerate(RELATIONSHIP_HORIZONS):
    row = position // 2 + 1
    column = position % 2 + 1
    target = target_columns[horizon - 1]
    change = features[target] - features["water_level"]
    midpoint, count = binned_distribution(change, bins=70, upper_quantile=None)
    target_change_figure.add_trace(
        go.Bar(
            x=midpoint,
            y=count,
            marker_color=PLOT_COLORS["water"],
            showlegend=False,
        ),
        row=row,
        col=column,
    )
    target_change_figure.add_vline(
        x=0, line_color="#333", line_dash="dot", row=row, col=column
    )
    target_change_figure.update_xaxes(
        title_text="Future change (cm)", row=row, col=column
    )
    target_change_figure.update_yaxes(
        title_text="Valid issue times", row=row, col=column
    )
target_change_figure.update_layout(
    title="Target-change distributions at selected horizons",
    template=PLOT_TEMPLATE,
    height=700,
)
target_change_figure.show()

## 9. Stored feature-generation seam

The feature artifacts were generated independently on either side of one timestamp.
The concatenated data remain a single analysis history, but stored lag and rolling
predictors cannot reach backward across this seam, and stored targets cannot reach
forward across it. The table quantifies only this expected boundary effect; it does
not introduce a partition label or compare partitions.

In [ ]:
max_lookback = max(
    feature_manifest["configuration"]["lag_hours"]
    + feature_manifest["configuration"]["rolling_windows"]
)
post_seam = features["timestamp"].between(
    seam_timestamp,
    seam_timestamp + pd.Timedelta(hours=max_lookback - 1),
)
pre_seam = features["timestamp"].between(
    seam_timestamp - pd.Timedelta(hours=horizon_hours),
    seam_timestamp,
    inclusive="left",
)
seam_sensitive_predictors = [
    column
    for column in predictor_columns
    if "lag_" in column or "rolling_" in column or column.startswith("imputed_count_")
]
seam_predictor_nulls = int(
    features.loc[post_seam, seam_sensitive_predictors].isna().sum().sum()
)
seam_preceding_invalid = int((~features.loc[pre_seam, "target_valid"]).sum())
seam_summary = pd.DataFrame(
    {
        "metric": [
            "Internal generation seam (UTC)",
            "Maximum stored lookback (hours)",
            "Seam-sensitive predictor null cells in following lookback window",
            "Target-invalid issue rows in preceding horizon",
            "Rows changed or recomputed by this notebook",
        ],
        "value": [
            utc_text(seam_timestamp),
            max_lookback,
            seam_predictor_nulls,
            seam_preceding_invalid,
            0,
        ],
    }
)
display(seam_summary)

## 10. Dynamically generated findings

The final narrative below is calculated from the validated artifacts and statistical
outputs above. It updates automatically if the inputs are regenerated under the same
contracts.

In [ ]:
def combined_stationarity_label(transformation: str) -> str:
    """Combine ADF and KPSS decisions into a concise finding."""
    subset = stationarity_results.loc[
        stationarity_results["transformation"].eq(transformation)
    ]
    adf_p = subset.loc[subset["test"].str.startswith("ADF"), "p_value"].iloc[0]
    kpss_p = subset.loc[subset["test"].str.startswith("KPSS"), "p_value"].iloc[0]
    if adf_p < 0.05 and kpss_p >= 0.05:
        return "stationary by both 5% decisions"
    if adf_p >= 0.05 and kpss_p < 0.05:
        return "non-stationary by both 5% decisions"
    return "mixed/inconclusive across ADF and KPSS at 5%"


strongest = strongest_relationships.iloc[0]
peak = episodes.iloc[0]
total_years = (
    processed["timestamp"].iloc[-1] - processed["timestamp"].iloc[0]
).total_seconds() / (365.25 * 24 * 3600)
raw_coverage_pct = 100 * raw_observed.mean()
processed_coverage_pct = 100 * processed["water_level"].notna().mean()
imputed_pct = 100 * processed["imputed"].mean()
target_valid_pct = 100 * features["target_valid"].mean()
constant_predictors = int(predictor_health["constant_non_null"].sum())

findings = f"""
### Coverage and integrity

The validated full history contains **{len(processed):,} hourly UTC rows** from
**{processed["timestamp"].iloc[0]:%Y-%m-%d %H:%M}** to
**{processed["timestamp"].iloc[-1]:%Y-%m-%d %H:%M}** ({total_years:.1f} years).
Raw gauge coverage is **{raw_coverage_pct:.2f}%** and processed water-level
coverage is **{processed_coverage_pct:.2f}%**. All manifest hashes, schemas,
timestamp ranges, row alignments, source-column preservation checks, and target
contracts passed.

### Gaps, preprocessing, and weather quality

The raw gauge grid contains **{len(raw_gap_runs):,} missing episodes** and
**{int(raw_grid_missing.sum()):,} missing hours**; the longest gap is
**{int(raw_gap_runs["hours"].max()):,} hours**. Interpolation filled and flagged
**{int(processed["imputed"].sum()):,} hours ({imputed_pct:.3f}%)** only in runs of
at most {MAX_INTERPOLATION_GAP_HOURS} hours. **{int(processed["water_level"].isna().sum()):,}
hours** remain absent in longer gaps. Weather has
**{int(processed["precipitation"].isna().sum()):,} precipitation** and
**{int(processed["temperature_2m"].isna().sum()):,} temperature** nulls;
**{len(negative_precipitation):,} negative precipitation values** are reported as
source anomalies and are not altered.

### Extremes and threshold episodes

The catalog alarm threshold is **{alarm_threshold_cm:.0f} cm**. There are
**{len(episodes):,} consecutive exceedance episodes**. The largest peaks at
**{peak.peak_cm:.0f} cm** on **{peak.peak_utc:%Y-%m-%d %H:%M UTC}**, lasts
**{int(peak.duration_hours)} hours**, and follows
**{peak.preceding_rain_24h_mm:.1f} mm / {peak.preceding_rain_72h_mm:.1f} mm** of
rain in the preceding 24 / 72 hours.

### Time-series structure and weather relationships

On the longest uninterrupted observed segment
({int(longest_observed_run["hours"]):,} hours), the level series is
**{combined_stationarity_label("Level")}**, while its first difference is
**{combined_stationarity_label("First difference")}**. The strongest tested
weather association is **{strongest.weather_aggregate}** with the
**t+{int(strongest.horizon_hours)} h** change
(Spearman ρ = **{strongest.spearman_rho:.3f}**, n =
**{int(strongest.paired_rows):,}**). These full-history associations are descriptive
and do not establish causality.

### Feature redundancy, target readiness, and seam effects

Across {len(predictor_columns)} predictors, there are
**{len(high_correlation_pairs):,} pairs with |r| > 0.95**,
**{constant_predictors} constant predictors**, and no infinite predictor values.
Complete 24-hour target vectors are available at **{target_valid_pct:.2f}%** of
issue times. Independent artifact generation creates
**{seam_predictor_nulls:,} seam-sensitive predictor null cells** in the following
{max_lookback}-hour lookback window and **{seam_preceding_invalid:,} target-invalid
rows** in the preceding {horizon_hours}-hour horizon; no stored value was recomputed
or changed here.

### Modeling implications

Models should handle the remaining long water-level gaps and rare weather nulls
explicitly, respect chronological evaluation, fit transformations or differenced
formulations where stationarity assumptions require them, and control the strong
multicollinearity among lag/rolling predictors (for example through regularization
or feature selection). Threshold events are rare relative to ordinary conditions,
so event-aware error analysis is warranted. Because this notebook combines all
rows, none of these findings constitutes blind test evidence.
"""
display(Markdown(findings))